In [1]:
import pandas as pd
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split

df = fetch_openml("credit-g", version = 1, as_frame = True).frame

X = df.drop(columns="class")
y = (df["class"] == "bad").astype(int) #Gives the output: 1 (bad) or 0 (g00d). :D

print(X.shape, y.shape)
print(y.value_counts())

(1000, 20) (1000,)
class
0    700
1    300
Name: count, dtype: int64


In [2]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.3, random_state = 42, stratify = y)

print(X_train.shape, X_test.shape)
print(y_train.mean(), y_test.mean())

(700, 20) (300, 20)
0.3 0.3


**Train/Test split** - Dataset split into 70/30, 700 training (70%) and 300 testing (30%), similar to 20%/80% split, just with a higher form of focusing on "testing" rather than onto training. It uses stratify to properly make a proportion between the two numbers and we have done a random_state of 42, as the "Guide to the Galaxy" proposes :D. The test stays hidden while model learns, so it doesn't get to "data leakage" and for us to fail the experiment. "making the result look better than they really are" ~ inaccuracies with the exp.    

In [3]:
cat_cols = X_train.select_dtypes(include="category").columns.tolist()
num_cols = X_train.select_dtypes(include="number").columns.tolist()

print("Categorical: ", cat_cols)
print("Numeric: ", num_cols)
print(len(cat_cols), len(num_cols))

Categorical:  ['checking_status', 'credit_history', 'purpose', 'savings_status', 'employment', 'personal_status', 'other_parties', 'property_magnitude', 'other_payment_plans', 'housing', 'job', 'own_telephone', 'foreign_worker']
Numeric:  ['duration', 'credit_amount', 'installment_commitment', 'residence_since', 'age', 'existing_credits', 'num_dependents']
13 7


In [4]:
X_train["checking_status"].head()

10        0<=X<200
82     no checking
827    no checking
410       0<=X<200
48     no checking
Name: checking_status, dtype: category
Categories (4, str): ['0<=X<200', '<0', '>=200', 'no checking']

In [5]:
pd.get_dummies(X_train["checking_status"]).head()

,0<=X<200,<0,>=200,no checking
10,True,False,False,False
82,False,False,False,True
827,False,False,False,True
410,True,False,False,False
48,False,False,False,True


In [6]:
pd.get_dummies(X_train["checking_status"], dtype=int).head()

,0<=X<200,<0,>=200,no checking
10,1,0,0,0
82,0,0,0,1
827,0,0,0,1
410,1,0,0,0
48,0,0,0,1


In [9]:
pd.get_dummies(X_train["checking_status"], dtype=int).sum(axis=1).head()

10     1
82     1
827    1
410    1
48     1
dtype: int64

**One-hot Encoding** - 13 categorical and 7 numerical columns. Categorical columns must be encoded because models only read in numbers (of course). Numbering a category, for instance (1,2,3,4 and so on) would create a confusion, a fake order and a model could get confused, thinking of one as a higher or more worthy than the other. One column per category, 1 if the borrower belongs to a category and 0 if it doesn't (no repeating with the numbers per row). Each row, as this last evidence of a result of code proves - all row sums equal 1.

In [12]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

preprocessor = ColumnTransformer (transformers=[
  ("num", StandardScaler(), num_cols),
  ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), cat_cols)
  ]
)

X_train_prep = preprocessor.fit_transform(X_train) #We use fit for learning (on training) and transform also, so it applies, what he has learned.
X_test_prep = preprocessor.transform(X_test) #When we test, we use only transform (so transform 2 times, because of applying fit - learning on training and applying transform on testing).

print(X_train_prep.shape, X_test_prep.shape)

preprocessor.get_feature_names_out()[:12]

(700, 61) (300, 61)


array(['num__duration', 'num__credit_amount',
       'num__installment_commitment', 'num__residence_since', 'num__age',
       'num__existing_credits', 'num__num_dependents',
       'cat__checking_status_0<=X<200', 'cat__checking_status_<0',
       'cat__checking_status_>=200', 'cat__checking_status_no checking',
       'cat__credit_history_all paid'], dtype=object)

In [13]:
for col in cat_cols:
    print(col, X_train[col].nunique())

X_train[cat_cols].nunique().sum()

checking_status 4
credit_history 5
purpose 10
savings_status 5
employment 5
personal_status 4
other_parties 3
property_magnitude 4
other_payment_plans 3
housing 3
job 4
own_telephone 2
foreign_worker 2


np.int64(54)

**Preprocessing:** Numeric columns (7) are scaled with StandardScaler; categorical
columns (13) are one-hot encoded with OneHotEncoder, creating 54 columns (one per
category). Total: 7 + 54 = 61 columns. The preprocessor is fit only on the training
data and then applied to both sets, to avoid data leakage (transform for applying - that is why it is on both).